# ETL Process: Data Warehouse Implementation

This notebook implements the complete ETL process to transform cleaned source data into a star schema data warehouse.

## Process Overview
1. **Extract**: Load cleaned source data
2. **Transform**: Create dimensional and fact tables
3. **Validate**: Quality checks and referential integrity
4. **Load**: Export to CSV and generate reports

## Setup: Imports and Configuration

In [49]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_execution.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
SOURCE_DIR = "."
OUTPUT_DIR = "./dw_output"
DIMS_DIR = os.path.join(OUTPUT_DIR, "dimensions")
FACTS_DIR = os.path.join(OUTPUT_DIR, "facts")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")

# Create output directories
for dir_path in [DIMS_DIR, FACTS_DIR, LOGS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

logger.info("ETL Process Started")
logger.info(f"Output directories created: {OUTPUT_DIR}")

2026-04-04 22:13:54,972 - INFO - ETL Process Started
2026-04-04 22:13:54,973 - INFO - Output directories created: ./dw_output


## Phase 1: Extraction

Load all cleaned source datasets

In [50]:
# Load raw source data
logger.info("=" * 60)
logger.info("PHASE 1: EXTRACTION & CLEANING")
logger.info("=" * 60)

try:
    # Load raw files
    df_pop_raw = pd.read_csv("ine_population_data.csv", encoding="latin1", sep=";", skiprows=3)
    logger.info(f"✓ Population data extracted: {len(df_pop_raw)} rows")
    
    df_substations_raw = pd.read_csv("postos-transformacao-distribuicao.csv", sep=";")
    logger.info(f"✓ Substations data extracted: {len(df_substations_raw)} rows")
    
    df_weather_sp_raw = pd.read_csv("porto_serra_do_pilar_weather.csv")
    logger.info(f"✓ Weather (Serra do Pilar) extracted: {len(df_weather_sp_raw)} rows")
    
    df_weather_pr_raw = pd.read_csv("porto_pedras_rubras_weather.csv")
    logger.info(f"✓ Weather (Pedras Rubras) extracted: {len(df_weather_pr_raw)} rows")
    
    df_consumption_pt1_raw = pd.read_csv("consumo_horario_mobilidade_eletrica_01_a_10.csv", sep=";")
    df_consumption_pt2_raw = pd.read_csv("consumo_horario_mobilidade_eletrica_11_a_18.csv", sep=";")
    logger.info(f"✓ Consumption data extracted: {len(df_consumption_pt1_raw) + len(df_consumption_pt2_raw)} rows")
    
    # Raw charging data will be loaded with proper encoding
    df_charging_raw = pd.read_csv(
        "ine_charging_points_number_location.csv", 
        encoding='iso-8859-1',
        sep=';',
        on_bad_lines='skip',
        engine='python'
    )
    logger.info(f"✓ Raw Charging data loaded: {len(df_charging_raw)} rows")
    
    logger.info("Extraction completed successfully")
except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
    raise
except Exception as e:
    logger.error(f"Error during extraction: {e}")
    raise

2026-04-04 22:13:55,001 - INFO - ============================================================
2026-04-04 22:13:55,003 - INFO - PHASE 1: EXTRACTION & CLEANING
2026-04-04 22:13:55,005 - INFO - ============================================================
2026-04-04 22:13:55,032 - INFO - ✓ Population data extracted: 180 rows
2026-04-04 22:13:55,262 - INFO - ✓ Substations data extracted: 72349 rows
2026-04-04 22:13:55,267 - INFO - ✓ Weather (Serra do Pilar) extracted: 1520 rows
2026-04-04 22:13:55,270 - INFO - ✓ Weather (Pedras Rubras) extracted: 1520 rows
2026-04-04 22:14:06,465 - INFO - ✓ Consumption data extracted: 9315214 rows
2026-04-04 22:14:06,475 - INFO - ✓ Raw Charging data loaded: 7 rows
2026-04-04 22:14:06,476 - INFO - Extraction completed successfully


## 1.1: Clean Population Data

In [51]:
# Clean population data
df_pop = df_pop_raw.copy()
df_pop.drop(0, inplace=True)

# Rename columns
df_pop.columns = [
    "year", "region", "growth_rate", "density", 
    "cities", "freguesias", "vilas", "extra"
]

# Fill year forward (for merged cells in original data)
df_pop["year"] = df_pop["year"].ffill()

# Select data of interest (first 73 rows contain Porto Metropolitan Area data)
df_pop = df_pop.iloc[0:72]

# Split region into code and name
df_pop[["region_code", "region_name"]] = df_pop["region"].str.split(":", n=1, expand=True)
df_pop = df_pop.drop('region', axis=1)

# Remove extra column and clean region names
df_pop = df_pop.drop('extra', axis=1)
df_pop["region_name"] = df_pop["region_name"].str.strip()

# **IMPORTANT**: Rename region_code and region_name to municipality_code and municipality_name
df_pop = df_pop.rename(columns={"region_code": "municipality_code", "region_name": "municipality_name"})

# **IMPORTANT**: Convert numeric columns - handle Portuguese decimal separator (comma)
df_pop["year"] = df_pop["year"].astype(int)

# Replace commas with dots for decimal numbers (Portuguese format to standard format)
df_pop["growth_rate"] = df_pop["growth_rate"].astype(str).str.replace(",", ".", regex=False)
df_pop["growth_rate"] = pd.to_numeric(df_pop["growth_rate"], errors='coerce')

df_pop["density"] = df_pop["density"].astype(str).str.replace(",", ".", regex=False)
df_pop["density"] = pd.to_numeric(df_pop["density"], errors='coerce')

# Drop only truly unnecessary columns
df_pop = df_pop.drop(['cities', 'freguesias', 'vilas'], axis=1)

# Create municipality_map for consistent code-name mapping across all datasets
municipality_map = dict(
    df_pop[["municipality_name", "municipality_code"]]
    .drop_duplicates()
    .values
)

# Get unique regions for filtering other datasets
unique_regions = df_pop["municipality_name"].dropna().unique().tolist()

logger.info(f"✓ Population data cleaned: {len(df_pop)} rows")
logger.info(f"  Columns: {df_pop.columns.tolist()}")
logger.info(f"  Regions: {len(unique_regions)} municipalities")
logger.info(f"  Density non-null values: {df_pop['density'].notna().sum()}")
logger.info(f"  Growth rate non-null values: {df_pop['growth_rate'].notna().sum()}")

2026-04-04 22:14:06,509 - INFO - ✓ Population data cleaned: 72 rows
2026-04-04 22:14:06,509 - INFO -   Columns: ['year', 'growth_rate', 'density', 'municipality_code', 'municipality_name']
2026-04-04 22:14:06,510 - INFO -   Regions: 18 municipalities
2026-04-04 22:14:06,511 - INFO -   Density non-null values: 72
2026-04-04 22:14:06,513 - INFO -   Growth rate non-null values: 72


In [52]:
df_pop

,year,growth_rate,density,municipality_code,municipality_name
1,2024,0.86,890.7,11A,Área Metropolitana do Porto
2,2024,-0.43,63.3,11A0104,Arouca
3,2024,1.05,1554.4,11A0107,Espinho
4,2024,0.48,1284.0,11A1304,Gondomar
5,2024,1.44,1744.0,11A1306,Maia
...,...,...,...,...,...
68,2021,0.88,541.7,11A1318,Trofa
69,2021,-0.26,144.5,11A0119,Vale de Cambra
70,2021,1.70,1286.6,11A1315,Valongo
71,2021,1.03,550.8,11A1316,Vila do Conde


## 1.2: Clean Substations Data

In [53]:
# Clean substations data - filter for Porto Metropolitan Area
# Strip whitespace from Municipality column for proper matching
df_substations_raw["Municipality"] = df_substations_raw["Municipality"].str.strip()

# Filter for Porto Metropolitan Area
df_substations = df_substations_raw[
    df_substations_raw["Municipality"].isin(unique_regions)
].copy()

logger.info(f"✓ Substations data cleaned: {len(df_substations)} rows")
logger.info(f"  Filtered to Porto Metropolitan Area")

2026-04-04 22:14:06,639 - INFO - ✓ Substations data cleaned: 8412 rows
2026-04-04 22:14:06,640 - INFO -   Filtered to Porto Metropolitan Area


In [54]:
df_substations

,Installation Code,District,Municipality,Geographic Coordinates,Installed Power [kVA],Usage Level [%],Construction Type,Contracted Power [kVA],Number of Clients,Power Generation [kW],Number of Client Producers,Usage Histogram,DistrictCode,DistrictMunicipalityCode
28,1312D2103700,Porto,Porto,"41.183677856847204, -8.608945507725844",630,0%-19%,Cabine metálica (monobloco),1317.35,201,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;5359....,13,1312
35,1316D2004400,Porto,Vila do Conde,"41.336527228320506, -8.645671024999814",400,20%-39%,Cabine alta,1610.01,216,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;3.25;...,13,1316
36,1316D2016800,Porto,Vila do Conde,"41.27164654205575, -8.724611320666718",630,20%-39%,Cabine baixa em edifício próprio,2047.95,336,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;7.75;...,13,1316
37,1316D2057000,Porto,Vila do Conde,"41.279717670991126, -8.679046573660683",400,0%-19%,Cabine pré-fabricada,N/D,<20,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;1.5;8290....,13,1316
39,1317D2070300,Porto,Vila Nova de Gaia,"41.04344892828184, -8.542996018121055",250,20%-39%,Cabine baixa em edifício próprio,640.00,71,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;575.5...,13,1317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72317,0113D2008400,Aveiro,Oliveira de Azeméis,"40.78512433492829, -8.467140436977383",400,20%-39%,Cabine pré-fabricada,1089.14,153,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;910.5...,1,113
72318,0113D2007900,Aveiro,Oliveira de Azeméis,"40.890482619464066, -8.513090092169527",400,20%-39%,Cabine baixa em edifício próprio,1382.89,188,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;4119....,1,113
72319,0113D2025000,Aveiro,Oliveira de Azeméis,"40.78321708858559, -8.482865377918259",250,20%-39%,Cabine baixa em edifício próprio,513.32,50,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;1802....,1,113
72320,0113D2010000,Aveiro,Oliveira de Azeméis,"40.84480155852852, -8.460583166128844",400,0%-19%,Cabine alta,N/D,<20,N/D,<20,(0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;0.0;8741....,1,113


## 1.3: Clean Weather Data

In [55]:
# Clean weather data for both stations
for df_raw, station_name in [(df_weather_sp_raw, "Serra do Pilar"), 
                              (df_weather_pr_raw, "Pedras Rubras")]:
    # Convert date column
    df_raw["date"] = pd.to_datetime(df_raw["date"])
    
    # Extract datetime components
    df_raw["year"] = df_raw["date"].dt.year
    df_raw["month"] = df_raw["date"].dt.month
    df_raw["day"] = df_raw["date"].dt.day
    df_raw["time"] = df_raw["date"].dt.time
    
    # Drop columns with all null values
    df_raw.drop(columns=["snow", "wdir", "tsun"], inplace=True)
    
    logger.info(f"  Weather ({station_name}) cleaned: {len(df_raw)} rows")

# Assign cleaned data
df_weather_sp = df_weather_sp_raw
df_weather_pr = df_weather_pr_raw

logger.info(f"✓ Weather data cleaned for both stations")

2026-04-04 22:14:06,703 - INFO -   Weather (Serra do Pilar) cleaned: 1520 rows
2026-04-04 22:14:06,710 - INFO -   Weather (Pedras Rubras) cleaned: 1520 rows
2026-04-04 22:14:06,712 - INFO - ✓ Weather data cleaned for both stations


In [56]:
df_weather_sp

,date,tavg,tmin,tmax,prcp,wspd,wpgt,pres,year,month,day,time
0,2022-01-01,16.4,11.6,23.7,0.0,14.9,29.6,1025.3,2022,1,1,00:00:00
1,2022-01-02,13.8,10.9,23.7,4.3,7.4,24.1,1029.7,2022,1,2,00:00:00
2,2022-01-03,11.9,8.8,17.2,0.3,13.4,33.3,1025.5,2022,1,3,00:00:00
3,2022-01-04,12.1,8.5,15.3,9.7,14.7,44.5,1018.4,2022,1,4,00:00:00
4,2022-01-05,9.7,7.1,13.3,10.4,8.9,31.5,1022.3,2022,1,5,00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
1515,2026-02-24,13.0,10.3,17.4,0.0,13.1,31.5,1017.1,2026,2,24,00:00:00
1516,2026-02-25,12.6,10.4,14.3,4.6,9.7,27.8,1017.1,2026,2,25,00:00:00
1517,2026-02-26,11.6,9.1,16.4,0.0,7.1,25.9,1024.5,2026,2,26,00:00:00
1518,2026-02-27,9.9,7.5,12.5,3.3,10.3,25.9,1025.4,2026,2,27,00:00:00


## 1.4: Clean Consumption Data

In [57]:
# Clean consumption data - combine and filter for Porto Metropolitan Area
df_consumption_pt1 = df_consumption_pt1_raw[
    df_consumption_pt1_raw["Municipality"].isin(unique_regions)
]
df_consumption_pt2 = df_consumption_pt2_raw[
    df_consumption_pt2_raw["Municipality"].isin(unique_regions)
]

# Combine both parts
df_consumption = pd.concat(
    [df_consumption_pt1, df_consumption_pt2], 
    axis=0, 
    ignore_index=True
)

# **IMPORTANT**: Standardize consumption data column names and add municipality codes
# Rename key columns to match the cleaned dataset format
df_consumption = df_consumption.rename(columns={
    "Date/Time": "datetime",
    "Date": "date",
    "Hour": "hour",
    "District": "district_name",
    "Parish": "parish_name",
    "Parish Code": "parish_code",
    "Energy Consumed": "energy_consumed",
    "District Code": "district_code",
})

# Strip whitespace from Municipality column and map to municipality_code
df_consumption["Municipality"] = df_consumption["Municipality"].str.strip()
df_consumption["municipality_code"] = df_consumption["Municipality"].map(municipality_map)
df_consumption = df_consumption.rename(columns={"Municipality": "municipality_name"})

# Drop redundant Municipality Code column if exists
df_consumption = df_consumption.drop(columns=["Municipality Code"], errors="ignore")
df_consumption = df_consumption.drop(columns=["group_name"], errors="ignore")

# Convert datetime
df_consumption["datetime"] = pd.to_datetime(df_consumption.get("datetime", df_consumption.get("date")), errors="coerce", utc=True)

logger.info(f"✓ Consumption data cleaned: {len(df_consumption)} rows")
logger.info(f"  Combined from 2 datasets and filtered to Porto Metropolitan Area")

2026-04-04 22:14:10,938 - INFO - ✓ Consumption data cleaned: 1027232 rows
2026-04-04 22:14:10,939 - INFO -   Combined from 2 datasets and filtered to Porto Metropolitan Area


In [58]:
df_consumption

,datetime,date,hour,district_name,municipality_name,parish_name,energy_consumed,district_code,parish_code,Dia,Mês,Ano,municipality_code
0,2025-10-01 04:00:00+00:00,2025-10-01,04:00,Aveiro,Santa Maria da Feira,Milheirós de Poiares,0.00,1,010914,1,10,2025,11A0109
1,2025-10-04 23:00:00+00:00,2025-10-04,23:00,Aveiro,Santa Maria da Feira,Milheirós de Poiares,0.00,1,010914,4,10,2025,11A0109
2,2025-10-26 23:00:00+00:00,2025-10-26,23:00,Aveiro,Santa Maria da Feira,Milheirós de Poiares,0.00,1,010914,26,10,2025,11A0109
3,2025-10-22 03:00:00+00:00,2025-10-22,03:00,Aveiro,Santa Maria da Feira,Milheirós de Poiares,0.00,1,010914,22,10,2025,11A0109
4,2025-10-10 11:00:00+00:00,2025-10-10,11:00,Aveiro,Santa Maria da Feira,Milheirós de Poiares,0.00,1,010914,10,10,2025,11A0109
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1027227,2026-01-16 07:00:00+00:00,2026-01-16,07:00,Porto,Vila do Conde,"Bagunte, Ferreiró, Outeiro Maior e Parada",0.00,13,131631,16,1,2026,11A1316
1027228,2026-01-22 15:00:00+00:00,2026-01-22,15:00,Porto,Vila do Conde,"Bagunte, Ferreiró, Outeiro Maior e Parada",1.07,13,131631,22,1,2026,11A1316
1027229,2026-01-30 07:00:00+00:00,2026-01-30,07:00,Porto,Vila Nova de Gaia,Avintes,43.40,13,131702,30,1,2026,11A1317
1027230,2026-01-02 14:00:00+00:00,2026-01-02,14:00,Porto,Vila do Conde,"Bagunte, Ferreiró, Outeiro Maior e Parada",0.00,13,131631,2,1,2026,11A1316


## 1.5: Clean Charging Data

In [59]:
def clean_charging_data(filename):
    """
    Clean the raw INE charging points data.
    
    File structure has multi-row headers:
    - Row 2 (index 1): Period (month names with years)
    - Row 4 (index 3): Outlet type (socket types)
    - Row 6+ (index 5+): Data rows (location in col 0, values in cols 1+)
    
    Uses pandas melt() to transform from wide to long format, ensuring all values are preserved.
    """
    
    # Read raw data
    df = pd.read_csv(
        filename,
        encoding="latin1",
        sep=";",
        skiprows=3
    )
    
    # Extract header rows before removing
    row_period = df.iloc[1]
    row_type = df.iloc[3]
    
    # Build column names from period and outlet type rows
    columns = []
    current_period = None
    
    for i in range(len(df.columns)):
        if pd.notna(row_period.iloc[i]):
            current_period = row_period.iloc[i]
        
        socket_type = row_type.iloc[i]
        
        if i == 0:
            columns.append("region")
        elif pd.notna(socket_type):
            columns.append(f"{current_period} | {socket_type}")
        else:
            columns.append(None)
    
    df.columns = columns
    
    # Remove header rows
    df = df.iloc[5:].reset_index(drop=True)
    
    # Remove empty columns
    df = df.loc[:, df.columns.notna()]
    
    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Replace missing value marker
    df = df.replace("- -", pd.NA)
    
    # Select only municipality rows (first 18 rows)
    df = df.iloc[0:18]
    
    # Split region into code and name
    df[["code", "region"]] = df["region"].str.split(": ", expand=True)
    
    # Melt from wide to long format
    df_long = df.melt(
        id_vars=["code", "region"],
        var_name="Period_Type",
        value_name="Value"
    )
    
    # Split Period and Outlet_Type from combined string
    df_long[["Period", "Outlet_Type"]] = df_long["Period_Type"].str.split(
        " | ", expand=True, regex=False
    )
    df_long = df_long.drop('Period_Type', axis=1)
    
    # Extract year and month
    df_long["year"] = df_long["Period"].str.extract(r"(\d{4})").astype(int)
    df_long["month_name"] = df_long["Period"].str.extract(r"^([A-Za-zçÇ]+)")
    
    # Map month names to numbers
    month_map = {
        "Janeiro": 1, "Fevereiro": 2, "Março": 3, "Abril": 4,
        "Maio": 5, "Junho": 6, "Julho": 7, "Agosto": 8,
        "Setembro": 9, "Outubro": 10, "Novembro": 11, "Dezembro": 12
    }
    df_long["month"] = df_long["month_name"].map(month_map)
    
    # Drop temporary columns
    df_long = df_long.drop(columns=["month_name", "Period"])
    
    # Rename to final column names
    df_charged = df_long.rename(columns={
        "code": "municipality_code",
        "region": "municipality_name",
        "Value": "value",
        "Outlet_Type": "outlet_type"
    })
    
    # Create year_id and month_id
    df_charged["year_id"] = df_charged["year"].astype(int) % 100
    df_charged["month_id"] = df_charged["year"].astype(str) + df_charged["month"].astype(str).str.zfill(2)
    df_charged["month_id"] = df_charged["month_id"].astype(int)
    
    # Convert value to numeric
    df_charged["value"] = pd.to_numeric(df_charged["value"], errors='coerce')
    
    return df_charged

# Clean the raw charging data
df_charging = clean_charging_data("ine_charging_points_number_location.csv")

logger.info(f"✓ Charging data cleaned: {len(df_charging)} rows")
if len(df_charging) > 0:
    logger.info(f"  Date range: {df_charging['year'].min()}-{df_charging['month'].min():02d} to {df_charging['year'].max()}-{df_charging['month'].max():02d}")
    logger.info(f"  Locations: {df_charging['municipality_name'].nunique()}")
    logger.info(f"  Outlet types: {len(df_charging['outlet_type'].unique())}")
    logger.info(f"  Missing values in 'value': {df_charging['value'].isna().sum()}")

2026-04-04 22:14:11,183 - INFO - ✓ Charging data cleaned: 4320 rows
2026-04-04 22:14:11,184 - INFO -   Date range: 2021-01 to 2024-12
2026-04-04 22:14:11,185 - INFO -   Locations: 18
2026-04-04 22:14:11,186 - INFO -   Outlet types: 5
2026-04-04 22:14:11,189 - INFO -   Missing values in 'value': 1088


## Phase 2: Transformation

### 2.1 Create DIM_LOCATION

In [60]:
logger.info("=" * 60)
logger.info("PHASE 2: TRANSFORMATION")
logger.info("=" * 60)

# **IMPORTANT**: Extract unique locations from CONSUMPTION data which has parish information
# This ensures DIM_LOCATION has complete geographic hierarchy
location_cols = ['municipality_code', 'municipality_name', 'parish_code', 'parish_name', 'district_code', 'district_name']
available_location_cols = [col for col in location_cols if col in df_consumption.columns]

dim_location = df_consumption[available_location_cols].drop_duplicates().reset_index(drop=True)

# Assign location_id (surrogate key)
dim_location['location_id'] = range(1, len(dim_location) + 1)

# Reorder columns according to schema
dim_location = dim_location[[
    'location_id',
    'municipality_code',
    'municipality_name',
    'parish_code',
    'parish_name',
    'district_code',
    'district_name'
]]

logger.info(f"✓ DIM_LOCATION created: {len(dim_location)} rows")
logger.info(f"  Unique municipalities: {dim_location['municipality_name'].nunique()}")
logger.info(f"  Unique parishes: {dim_location['parish_name'].nunique()}")
display(dim_location.head(10))

2026-04-04 22:14:11,203 - INFO - ============================================================
2026-04-04 22:14:11,206 - INFO - PHASE 2: TRANSFORMATION
2026-04-04 22:14:11,209 - INFO - ============================================================
2026-04-04 22:14:11,505 - INFO - ✓ DIM_LOCATION created: 130 rows
2026-04-04 22:14:11,506 - INFO -   Unique municipalities: 17
2026-04-04 22:14:11,507 - INFO -   Unique parishes: 130


,location_id,municipality_code,municipality_name,parish_code,parish_name,district_code,district_name
0,1,11A0109,Santa Maria da Feira,010914,Milheirós de Poiares,1,Aveiro
1,2,11A0109,Santa Maria da Feira,010904,Escapães,1,Aveiro
2,3,11A0107,Espinho,010706,Anta e Guetim,1,Aveiro
3,4,11A0109,Santa Maria da Feira,010925,Santa Maria de Lamas,1,Aveiro
4,5,11A0107,Espinho,010702,Espinho,1,Aveiro
5,6,11A0116,São João da Madeira,011601,São João da Madeira,1,Aveiro
6,7,11A0109,Santa Maria da Feira,010907,Fiães,1,Aveiro
7,8,11A0109,Santa Maria da Feira,010935,"Santa Maria da Feira, Travanca, Sanfins e Espargo",1,Aveiro
8,9,11A0104,Arouca,010421,Arouca e Burgo,1,Aveiro
9,10,11A0109,Santa Maria da Feira,010913,Lourosa,1,Aveiro


### 2.2 Create DIM_TIME

In [62]:
# Extract min and max dates from all datasets
dates = []

# From population: year
pop_dates = pd.to_datetime(df_pop['year'].astype(str))
dates.extend(pop_dates.tolist())

# From weather
if 'date' in df_weather_sp.columns:
    dates.extend(pd.to_datetime(df_weather_sp['date']).tolist())
if 'date' in df_weather_pr.columns:
    dates.extend(pd.to_datetime(df_weather_pr['date']).tolist())

# From consumption
if 'Period' in df_consumption.columns:
    dates.extend(pd.to_datetime(df_consumption['Period'], errors='coerce').dropna().tolist())

# From charging
charging_dates = pd.to_datetime(
    df_charging['year'].astype(str) + '-' + 
    df_charging['month'].astype(str) + '-01',
    errors='coerce'
)
dates.extend(charging_dates.tolist())

# Remove None and duplicates
dates = pd.to_datetime([d for d in dates if d is not None]).unique()
dates = sorted(dates)

min_date = pd.Timestamp(dates[0])
max_date = pd.Timestamp(dates[-1])

logger.info(f"Date range: {min_date.date()} to {max_date.date()}")

# Generate complete datetime range
date_range = pd.date_range(start=min_date, end=max_date, freq='H')

dim_time = pd.DataFrame({
    'datetime': date_range
})

# Extract datetime components
dim_time['date'] = dim_time['datetime'].dt.date
dim_time['year'] = dim_time['datetime'].dt.year
dim_time['month'] = dim_time['datetime'].dt.month
dim_time['day'] = dim_time['datetime'].dt.day
dim_time['hour'] = dim_time['datetime'].dt.hour

# Create level keys (surrogate keys for each level)
# Year level key
dim_time['time_id'] = dim_time['year'] - dim_time['year'].min() + 1

# Month level key
dim_time['month_id'] = (
    (dim_time['year'] - dim_time['year'].min()) * 12 + dim_time['month']
)

# Day level key
dim_time['day_id'] = (
    dim_time['datetime'].astype('int64') // (10**9 * 86400)
)

# Hour level key
dim_time['hour_id'] = range(len(dim_time))

# Reorder columns - PK first, then level keys, then attributes
dim_time = dim_time[[
    'time_id',              # PK - Primary Key (unique per hour)
    'hour_id',              # LK - Hour Level Key
    'day_id',               # LK - Day Level Key
    'month_id',             # LK - Month Level Key
    'hour',                 # Hour of day (0-23)
    'day',                  # Day of month (1-31)
    'month',                # Month (1-12)
    'year',                 # Year (YYYY)
    'date',                 # Date (YYYY-MM-DD)
    'datetime'              # Full timestamp (YYYY-MM-DD HH:MM:SS)
]]

logger.info(f"✓ DIM_TIME created: {len(dim_time)} rows")
logger.info(f"  Time range: {dim_time['datetime'].min()} to {dim_time['datetime'].max()}")
display(dim_time)

2026-04-04 22:14:11,675 - INFO - Date range: 2021-01-01 to 2026-02-28
C:\Users\iscap\AppData\Local\Temp\ipykernel_37032\3859189968.py:36: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range = pd.date_range(start=min_date, end=max_date, freq='H')
2026-04-04 22:14:11,714 - INFO - ✓ DIM_TIME created: 45217 rows
2026-04-04 22:14:11,716 - INFO -   Time range: 2021-01-01 00:00:00 to 2026-02-28 00:00:00


,time_id,hour_id,day_id,month_id,hour,day,month,year,date,datetime
0,1,0,18628,1,0,1,1,2021,2021-01-01,2021-01-01 00:00:00
1,1,1,18628,1,1,1,1,2021,2021-01-01,2021-01-01 01:00:00
2,1,2,18628,1,2,1,1,2021,2021-01-01,2021-01-01 02:00:00
3,1,3,18628,1,3,1,1,2021,2021-01-01,2021-01-01 03:00:00
4,1,4,18628,1,4,1,1,2021,2021-01-01,2021-01-01 04:00:00
...,...,...,...,...,...,...,...,...,...,...
45212,6,45212,20511,62,20,27,2,2026,2026-02-27,2026-02-27 20:00:00
45213,6,45213,20511,62,21,27,2,2026,2026-02-27,2026-02-27 21:00:00
45214,6,45214,20511,62,22,27,2,2026,2026-02-27,2026-02-27 22:00:00
45215,6,45215,20511,62,23,27,2,2026,2026-02-27,2026-02-27 23:00:00


### 2.3 Create DIM_WEATHER

In [70]:
# Combine weather data from both stations
df_weather_sp['date'] = pd.to_datetime(df_weather_sp['date'])
df_weather_pr['date'] = pd.to_datetime(df_weather_pr['date'])

# Merge both weather sources (average their values)
weather_merged = pd.merge(
    df_weather_sp[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    df_weather_pr[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    on='date',
    how='outer',
    suffixes=('_sp', '_pr')
)

# Calculate averages
dim_weather = pd.DataFrame({
    'date': weather_merged['date'],
    'avg_temperature': weather_merged[['tavg_sp', 'tavg_pr']].mean(axis=1),
    'min_temperature': weather_merged[['tmin_sp', 'tmin_pr']].min(axis=1),
    'max_temperature': weather_merged[['tmax_sp', 'tmax_pr']].max(axis=1),
    'precipitation': weather_merged[['prcp_sp', 'prcp_pr']].sum(axis=1),
    'wind_speed': weather_merged[['wspd_sp', 'wspd_pr']].mean(axis=1),
    'pressure': weather_merged[['pres_sp', 'pres_pr']].mean(axis=1)
})

# Link to time dimension (day level)
# Convert dim_time date to datetime64[ns] to match dim_weather for merge
dim_time_for_merge = dim_time[['date', 'day_id', 'month_id', 'time_id']].drop_duplicates().copy()
dim_time_for_merge['date'] = pd.to_datetime(dim_time_for_merge['date'])

dim_weather = dim_weather.merge(
    dim_time_for_merge,
    on='date',
    how='left'
)

# Assign weather_id
dim_weather['weather_id'] = range(1, len(dim_weather) + 1)
dim_weather.rename(columns={'time_id': 'time_id_ref'}, inplace=True)
#dim_weather['time_id'] = dim_weather['day_id']  # Reference day level key

# Reorder columns
dim_weather = dim_weather[[
    'weather_id',
    'day_id',
    'avg_temperature',
    'min_temperature',
    'max_temperature',
    'precipitation',
    'wind_speed',
    'pressure'
]]

logger.info(f"✓ DIM_WEATHER created: {len(dim_weather)} rows")
display(dim_weather.head(10))

2026-04-04 22:19:12,388 - INFO - ✓ DIM_WEATHER created: 1520 rows


,weather_id,day_id,avg_temperature,min_temperature,max_temperature,precipitation,wind_speed,pressure
0,1,18993,17.20,11.6,23.7,0.0,16.90,1024.55
1,2,18994,13.85,10.6,23.7,4.3,8.45,1029.25
2,3,18995,11.95,8.8,17.2,0.3,14.35,1024.90
3,4,18996,12.00,8.5,15.3,9.7,18.35,1017.75
4,5,18997,9.75,7.1,13.4,10.4,10.35,1021.70
5,6,18998,8.20,4.1,13.3,0.3,6.40,1024.35
6,7,18999,10.05,6.4,14.1,0.3,13.05,1031.45
7,8,19000,7.25,2.3,14.1,0.3,6.45,1032.85
8,9,19001,12.35,6.8,13.7,17.5,8.15,1028.25
9,10,19002,12.30,8.9,13.6,2.3,10.25,1026.70


### 2.4 Create DIM_SUBSTATIONS

In [71]:
# Process substations data
dim_substations = df_substations.copy()

# Assign substation_id (surrogate key)
dim_substations['substation_id'] = range(1, len(dim_substations) + 1)

# Rename and map columns to schema
dim_substations = dim_substations.rename(columns={
    'Installation Code': 'installation_code',
    'Municipality': 'municipality_name'
})

# Map to location dimension to get municipality_code
dim_substations = dim_substations.merge(
    dim_location[['municipality_code', 'municipality_name', 'district_name']],
    on='municipality_name',
    how='left'
)

# Assign level keys (using month_id as proxy for data level)
dim_substations['municipality_code_lk'] = dim_substations['municipality_code']

# Select and reorder columns
columns_to_keep = [
    'substation_id',
    'installation_code',
    'municipality_code_lk',
    'municipality_name',
    'district_name'
]

# Add optional columns if they exist
optional_cols = [
    'Geographic Coordinates',  # Geographic coordinates
    'Installed Power [kVA]',
    'Contracted Power [kVA]',
    'Usage Level [%]',
    'Number of Clients',
    'Construction Type'
]

for col in optional_cols:
    if col in dim_substations.columns:
        columns_to_keep.append(col)

dim_substations = dim_substations[columns_to_keep]

# Rename to match schema
rename_map = {
    'municipality_code_lk': 'municipality_code',
    'Geographic Coordinates': 'geographic_coordinates',
    'Installed Power [kVA]': 'installed_power',
    'Contracted Power [kVA]': 'contracted_power',
    'Usage Level [%]': 'usage_level',
    'Number of Clients': 'number_of_clients',
    'Construction Type': 'construction_type'
}

dim_substations = dim_substations.rename(columns=rename_map)

logger.info(f"✓ DIM_SUBSTATIONS created: {len(dim_substations)} rows")
display(dim_substations.head(10))

2026-04-04 22:19:28,993 - INFO - ✓ DIM_SUBSTATIONS created: 78501 rows


,substation_id,installation_code,municipality_code,municipality_name,district_name,geographic_coordinates,installed_power,contracted_power,usage_level,number_of_clients,construction_type
0,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
1,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
2,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
3,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
4,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
5,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
6,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
7,2,1316D2004400,11A1316,Vila do Conde,Porto,"41.336527228320506, -8.645671024999814",400,1610.01,20%-39%,216,Cabine alta
8,2,1316D2004400,11A1316,Vila do Conde,Porto,"41.336527228320506, -8.645671024999814",400,1610.01,20%-39%,216,Cabine alta
9,2,1316D2004400,11A1316,Vila do Conde,Porto,"41.336527228320506, -8.645671024999814",400,1610.01,20%-39%,216,Cabine alta


### 2.5 Create FACT_CONSUMPTION

In [77]:
# Process consumption data
fact_consumption = df_consumption.copy()

# Ensure datetime is properly formatted
fact_consumption['datetime'] = pd.to_datetime(fact_consumption['datetime'], errors='coerce', utc=True)
fact_consumption['datetime'] = fact_consumption['datetime'].dt.tz_localize(None)

# Map municipality_name to location_id (FK to Location dimension)
fact_consumption = fact_consumption.merge(
    dim_location[['municipality_name', 'location_id']],
    on='municipality_name',
    how='left'
)

# Map datetime to hour_id (FK to Time dimension - HOUR LEVEL using LK)
# According to schema: fact_consumption is HOURLY granularity, uses hour_id (lookup key) not day_id
if 'datetime' in fact_consumption.columns and pd.api.types.is_datetime64_any_dtype(fact_consumption['datetime']):
    fact_consumption = fact_consumption.merge(
        dim_time[['datetime', 'hour_id', 'day_id']],
        on='datetime',
        how='left'
    )
    
    # Extract date for weather matching (daily level)
    fact_consumption['date'] = fact_consumption['datetime'].dt.date
    
    # Map to weather_id (FK to Weather dimension - daily level)
    dim_weather_daily = dim_weather.merge(
        dim_time[['day_id', 'date']].drop_duplicates(),
        left_on='day_id',
        right_on='day_id',
        how='left'
    )[['weather_id', 'date']]
    
    fact_consumption = fact_consumption.merge(
        dim_weather_daily,
        on='date',
        how='left'
    )
    
    # Map to substation_id (FK to Substations dimension - municipality level)
    # Use the first substation per municipality to avoid cartesian product
    dim_substations_municipal = dim_substations[['substation_id', 'municipality_name', 'installed_power']].drop_duplicates(subset=['municipality_name'], keep='first')
    fact_consumption = fact_consumption.merge(
        dim_substations_municipal[['substation_id', 'municipality_name']],
        on='municipality_name',
        how='left'
    )

# Rename energy consumed column
if 'energy_consumed' not in fact_consumption.columns:
    if 'Energy Consumed' in fact_consumption.columns:
        fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['Energy Consumed'], errors='coerce')
    elif 'Value' in fact_consumption.columns:
        fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['Value'], errors='coerce')
    elif 'value' in fact_consumption.columns:
        fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['value'], errors='coerce')

# Select fact table columns matching the dimensional model
# hour_id is the HOUR-level lookup key for hourly granularity
fact_consumption = fact_consumption[[
    'location_id',
    'hour_id',
    'weather_id',
    'substation_id',
    'energy_consumed'
]]

# Remove rows with missing key references
fact_consumption = fact_consumption.dropna(subset=['location_id', 'hour_id', 'energy_consumed'])

In [73]:
fact_consumption

,location_id,day_id,weather_id,substation_id,energy_consumed
0,1,20362,1370,25,0.0
1,2,20362,1370,25,0.0
2,4,20362,1370,25,0.0
3,7,20362,1370,25,0.0
4,8,20362,1370,25,0.0
...,...,...,...,...,...
11159270,109,20473,1481,17,0.0
11159271,110,20473,1481,17,0.0
11159272,117,20473,1481,17,0.0
11159273,124,20473,1481,17,0.0


### 2.6 Create FACT_CHARGING

In [74]:
# Process charging data
fact_charging = df_charging.copy()

# Ensure municipality_name is properly formatted
fact_charging['municipality_name'] = fact_charging['municipality_name'].str.strip()

# Map municipality_name to location_id (FK to Location dimension)
fact_charging = fact_charging.merge(
    dim_location[['municipality_name', 'location_id']],
    on='municipality_name',
    how='left'
)

# Recalculate month_id to match dim_time's formula
# dim_time uses: month_id = (year_offset * 12 + month) where year_offset = year - min_year
min_year_time = dim_time['year'].min()
fact_charging['month_id_calc'] = (fact_charging['year'] - min_year_time) * 12 + fact_charging['month']

# Rename value to energy_consumed for consistency with fact table schema
if 'value' in fact_charging.columns:
    fact_charging['energy_consumed'] = pd.to_numeric(fact_charging['value'], errors='coerce')
elif 'charging_points' in fact_charging.columns:
    fact_charging['energy_consumed'] = pd.to_numeric(fact_charging['charging_points'], errors='coerce')

# Select fact table columns matching the dimensional model
# outlet_type is a degenerate dimension (stored directly in fact table)
# Use the calculated month_id that matches dim_time's LK calculation
fact_charging = fact_charging[[
    'location_id',
    'month_id_calc',
    'outlet_type',
    'energy_consumed'
]]

# Rename month_id_calc to month_id for final output
fact_charging.columns = ['location_id', 'month_id', 'outlet_type', 'energy_consumed']

# Remove rows with missing key references
fact_charging = fact_charging.dropna(subset=['location_id', 'month_id', 'energy_consumed'])

In [75]:
fact_charging

,location_id,month_id,outlet_type,energy_consumed
1,9.0,48,Total,9.0
2,16.0,48,Total,9.0
3,35.0,48,Total,9.0
4,40.0,48,Total,9.0
5,41.0,48,Total,9.0
...,...,...,...,...
31304,101.0,1,CCS,9.0
31305,105.0,1,CCS,9.0
31306,125.0,1,CCS,9.0
31307,128.0,1,CCS,9.0


### 2.7 Create FACT_POPULATION

In [76]:
# Process population data
fact_population = df_pop.copy()

# Remove rows with missing year or municipality_name
fact_population = fact_population.dropna(subset=['year', 'municipality_name'])

# **IMPORTANT**: Map to ALL location_ids for each municipality
# Since df_pop is at municipality level but dim_location is at parish level,
# we need to expand each population record to all parishes in that municipality
temp_locations = dim_location[['location_id', 'municipality_name']].drop_duplicates()
fact_population = fact_population.merge(
    temp_locations,
    on='municipality_name',
    how='left'
)

# Create datetime from year (01/01)
fact_population['datetime'] = pd.to_datetime(
    fact_population['year'].astype(str) + '-01-01'
)

# Map to year level time_id
fact_population = fact_population.merge(
    dim_time[['datetime', 'time_id']].drop_duplicates(),
    on='datetime',
    how='left'
)

# The density and growth_rate are already numeric from cleaning, but ensure they are
fact_population['density'] = pd.to_numeric(fact_population['density'], errors='coerce')
fact_population['growth_rate'] = pd.to_numeric(fact_population['growth_rate'], errors='coerce')

# Select fact table columns
fact_population = fact_population[[
    'location_id',
    'time_id',
    'density',
    'growth_rate'
]]

# Remove rows with missing key references
fact_population = fact_population.dropna(subset=['location_id', 'time_id'])

logger.info(f"✓ FACT_POPULATION created: {len(fact_population)} rows")
display(fact_population.head(10))

2026-04-04 22:21:29,978 - INFO - ✓ FACT_POPULATION created: 520 rows


,location_id,time_id,density,growth_rate
1,9.0,4,63.3,-0.43
2,16.0,4,63.3,-0.43
3,35.0,4,63.3,-0.43
4,40.0,4,63.3,-0.43
5,41.0,4,63.3,-0.43
6,3.0,4,1554.4,1.05
7,5.0,4,1554.4,1.05
8,31.0,4,1554.4,1.05
9,59.0,4,1284.0,0.48
10,65.0,4,1284.0,0.48


## Phase 3: Validation

Data quality checks and referential integrity

In [78]:
logger.info("=" * 60)
logger.info("PHASE 3: VALIDATION")
logger.info("=" * 60)

validation_report = []

def check_nulls(df, table_name, critical_columns):
    """Check for null values in critical columns"""
    null_count = df[critical_columns].isnull().sum()
    if null_count.sum() > 0:
        logger.warning(f"{table_name} - NULL values found:")
        for col, count in null_count[null_count > 0].items():
            logger.warning(f"  {col}: {count} nulls")
        validation_report.append(f"⚠ {table_name}: {null_count.sum()} null values")
    else:
        logger.info(f"✓ {table_name}: No null values in critical columns")
        validation_report.append(f"✓ {table_name}: No null values")

# Dimension validation
logger.info("\n--- DIMENSION VALIDATION ---")
check_nulls(dim_location, "DIM_LOCATION", ['location_id', 'municipality_code'])
check_nulls(dim_time, "DIM_TIME", ['time_id', 'year'])
check_nulls(dim_weather, "DIM_WEATHER", ['weather_id'])
check_nulls(dim_substations, "DIM_SUBSTATIONS", ['substation_id'])

# Fact validation
logger.info("\n--- FACT TABLE VALIDATION ---")
check_nulls(fact_consumption, "FACT_CONSUMPTION", ['location_id', 'hour_id', 'weather_id', 'energy_consumed'])
check_nulls(fact_charging, "FACT_CHARGING", ['location_id', 'month_id', 'outlet_type', 'energy_consumed'])
check_nulls(fact_population, "FACT_POPULATION", ['location_id', 'time_id'])

2026-04-04 22:23:17,249 - INFO - ============================================================
2026-04-04 22:23:17,252 - INFO - PHASE 3: VALIDATION
2026-04-04 22:23:17,254 - INFO - ============================================================


2026-04-04 22:23:17,261 - INFO - 
--- DIMENSION VALIDATION ---
2026-04-04 22:23:17,267 - INFO - ✓ DIM_LOCATION: No null values in critical columns
2026-04-04 22:23:17,272 - INFO - ✓ DIM_TIME: No null values in critical columns
2026-04-04 22:23:17,275 - INFO - ✓ DIM_WEATHER: No null values in critical columns
2026-04-04 22:23:17,279 - INFO - ✓ DIM_SUBSTATIONS: No null values in critical columns
2026-04-04 22:23:17,281 - INFO - 
--- FACT TABLE VALIDATION ---
2026-04-04 22:23:17,454 - INFO - ✓ FACT_CONSUMPTION: No null values in critical columns
2026-04-04 22:23:17,463 - INFO - ✓ FACT_CHARGING: No null values in critical columns
2026-04-04 22:23:17,464 - INFO - ✓ FACT_POPULATION: No null values in critical columns


In [79]:
# Referential Integrity Checks
logger.info("\n--- REFERENTIAL INTEGRITY ---")

def check_foreign_keys(fact_df, dim_df, fact_col, dim_col, table_names):
    """Check if all FK values exist in dimension"""
    missing = fact_df[~fact_df[fact_col].isin(dim_df[dim_col])]
    if len(missing) > 0:
        logger.warning(f"{table_names[0]} → {table_names[1]}: {len(missing)} orphan records")
        validation_report.append(f"⚠ {table_names[0]} → {table_names[1]}: {len(missing)} orphan FK")
    else:
        logger.info(f"✓ {table_names[0]} → {table_names[1]}: FK integrity OK")
        validation_report.append(f"✓ {table_names[0]} → {table_names[1]}: FK OK ({len(fact_df)} rows)")

# Check consumption FKs
check_foreign_keys(fact_consumption, dim_location, 'location_id', 'location_id', 
                   ['FACT_CONSUMPTION', 'DIM_LOCATION'])
check_foreign_keys(fact_consumption, dim_time, 'hour_id', 'hour_id', 
                   ['FACT_CONSUMPTION', 'DIM_TIME (hourly)'])
check_foreign_keys(fact_consumption, dim_weather, 'weather_id', 'weather_id', 
                   ['FACT_CONSUMPTION', 'DIM_WEATHER'])

# Check charging FKs
check_foreign_keys(fact_charging, dim_location, 'location_id', 'location_id', 
                   ['FACT_CHARGING', 'DIM_LOCATION'])
check_foreign_keys(fact_charging, dim_time, 'month_id', 'month_id', 
                   ['FACT_CHARGING', 'DIM_TIME (monthly)'])

# Check population FKs
check_foreign_keys(fact_population, dim_location, 'location_id', 'location_id', 
                   ['FACT_POPULATION', 'DIM_LOCATION'])
check_foreign_keys(fact_population, dim_time, 'time_id', 'time_id', 
                   ['FACT_POPULATION', 'DIM_TIME'])

2026-04-04 22:23:21,626 - INFO - 
--- REFERENTIAL INTEGRITY ---
2026-04-04 22:23:21,702 - INFO - ✓ FACT_CONSUMPTION → DIM_LOCATION: FK integrity OK
2026-04-04 22:23:21,748 - INFO - ✓ FACT_CONSUMPTION → DIM_TIME (hourly): FK integrity OK
2026-04-04 22:23:21,788 - INFO - ✓ FACT_CONSUMPTION → DIM_WEATHER: FK integrity OK
2026-04-04 22:23:21,791 - INFO - ✓ FACT_CHARGING → DIM_LOCATION: FK integrity OK
2026-04-04 22:23:21,793 - INFO - ✓ FACT_CHARGING → DIM_TIME (monthly): FK integrity OK
2026-04-04 22:23:21,795 - INFO - ✓ FACT_POPULATION → DIM_LOCATION: FK integrity OK
2026-04-04 22:23:21,796 - INFO - ✓ FACT_POPULATION → DIM_TIME: FK integrity OK


In [80]:
# Summary Statistics
logger.info("\n--- SUMMARY STATISTICS ---")

summary = f"""
DIM_LOCATION:        {len(dim_location):>7} rows | PKs: {len(dim_location['location_id'].unique())} unique
DIM_TIME:            {len(dim_time):>7} rows | PKs: {len(dim_time['time_id'].unique())} unique
DIM_WEATHER:         {len(dim_weather):>7} rows | PKs: {len(dim_weather['weather_id'].unique())} unique
DIM_SUBSTATIONS:     {len(dim_substations):>7} rows | PKs: {len(dim_substations['substation_id'].unique())} unique

FACT_CONSUMPTION:    {len(fact_consumption):>7} rows | Energy: {fact_consumption['energy_consumed'].sum():.2f} kWh
FACT_CHARGING:       {len(fact_charging):>7} rows | Energy: {fact_charging['energy_consumed'].sum():.2f} kWh
FACT_POPULATION:     {len(fact_population):>7} rows | Locations: {len(fact_population['location_id'].unique())}
"""

logger.info(summary)
validation_report.append(summary)

2026-04-04 22:24:49,546 - INFO - 
--- SUMMARY STATISTICS ---
2026-04-04 22:24:49,594 - INFO - 
DIM_LOCATION:            130 rows | PKs: 130 unique
DIM_TIME:              45217 rows | PKs: 6 unique
DIM_WEATHER:            1520 rows | PKs: 1520 unique
DIM_SUBSTATIONS:       78501 rows | PKs: 8412 unique

FACT_CONSUMPTION:    11159275 rows | Energy: 264288420.46 kWh
FACT_CHARGING:         23714 rows | Energy: 1023464.00 kWh
FACT_POPULATION:         520 rows | Locations: 130



## Phase 4: Loading

Export to CSV files

In [81]:
logger.info("=" * 60)
logger.info("PHASE 4: LOADING")
logger.info("=" * 60)

# Export Dimensions
logger.info("\nExporting dimension tables...")
dim_location.to_csv(os.path.join(DIMS_DIR, 'dim_location.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_location.csv")

dim_time.to_csv(os.path.join(DIMS_DIR, 'dim_time.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_time.csv")

dim_weather.to_csv(os.path.join(DIMS_DIR, 'dim_weather.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_weather.csv")

dim_substations.to_csv(os.path.join(DIMS_DIR, 'dim_substations.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_substations.csv")

# Export Facts
logger.info("\nExporting fact tables...")
fact_consumption.to_csv(os.path.join(FACTS_DIR, 'fact_consumption.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_consumption.csv")

fact_charging.to_csv(os.path.join(FACTS_DIR, 'fact_charging.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_charging.csv")

fact_population.to_csv(os.path.join(FACTS_DIR, 'fact_population.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_population.csv")

logger.info("\n✓ All tables exported successfully")

2026-04-04 22:24:53,044 - INFO - ============================================================
2026-04-04 22:24:53,046 - INFO - PHASE 4: LOADING
2026-04-04 22:24:53,047 - INFO - ============================================================
2026-04-04 22:24:53,048 - INFO - 
Exporting dimension tables...
2026-04-04 22:24:53,052 - INFO - ✓ ./dw_output\dimensions/dim_location.csv
2026-04-04 22:24:53,172 - INFO - ✓ ./dw_output\dimensions/dim_time.csv
2026-04-04 22:24:53,181 - INFO - ✓ ./dw_output\dimensions/dim_weather.csv
2026-04-04 22:24:53,365 - INFO - ✓ ./dw_output\dimensions/dim_substations.csv
2026-04-04 22:24:53,366 - INFO - 
Exporting fact tables...
2026-04-04 22:25:03,979 - INFO - ✓ ./dw_output\facts/fact_consumption.csv
2026-04-04 22:25:04,008 - INFO - ✓ ./dw_output\facts/fact_charging.csv
2026-04-04 22:25:04,012 - INFO - ✓ ./dw_output\facts/fact_population.csv
2026-04-04 22:25:04,012 - INFO - 
✓ All tables exported successfully


In [82]:
# Generate Validation Report
report_path = os.path.join(LOGS_DIR, 'etl_validation_report.txt')

with open(report_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("DATA WAREHOUSE ETL VALIDATION REPORT\n")
    f.write("="*60 + "\n\n")
    
    f.write(f"Execution Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("VALIDATION RESULTS:\n")
    f.write("-" * 60 + "\n")
    for item in validation_report:
        f.write(str(item) + "\n")
    
    f.write("\n" + "-" * 60 + "\n")
    f.write(f"ETL EXECUTION: SUCCESS\n")
    f.write(f"Output Directory: {OUTPUT_DIR}\n")

logger.info(f"✓ Validation report saved: {report_path}")
logger.info("="*60)
logger.info("ETL PROCESS COMPLETED SUCCESSFULLY")
logger.info("="*60)

2026-04-04 22:25:08,064 - INFO - ✓ Validation report saved: ./dw_output\logs\etl_validation_report.txt
2026-04-04 22:25:08,066 - INFO - ============================================================
2026-04-04 22:25:08,069 - INFO - ETL PROCESS COMPLETED SUCCESSFULLY
2026-04-04 22:25:08,072 - INFO - ============================================================


In [83]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables from .env file (located in parent directory)
env_path = Path.cwd().parent / '.env'
if not env_path.exists():
    env_path = Path.cwd() / '.env'
load_dotenv(env_path)

# Load to Database - Connect to MySQL and load data

logger.info("=" * 80)
logger.info("STEP 1: LOAD TO DATABASE - Connecting to MySQL and loading data")
logger.info("=" * 80)

# ============================================================================
# MYSQL CONFIGURATION - Load from .env file
# ============================================================================
MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = int(os.getenv('MYSQL_PORT', 3306))
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.getenv('MYSQL_DATABASE', 'data_warehouse')

logger.info(f"\nMySQL Connection Parameters:")
logger.info(f"  Host: {MYSQL_HOST}")
logger.info(f"  Port: {MYSQL_PORT}")
logger.info(f"  Database: {MYSQL_DATABASE}")
logger.info(f"  User: {MYSQL_USER}")

# Step 1: Connect to MySQL and create database if needed
logger.info("\nAttempting to connect to MySQL server...")

# First, connect to MySQL without specifying a database
root_connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}'

try:
    # Connect to MySQL server
    root_engine = create_engine(root_connection_string, echo=False)
    
    with root_engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    logger.info("Connected to MySQL server successfully!")
    
    # Create database if it doesn't exist
    logger.info(f"\nChecking if database '{MYSQL_DATABASE}' exists...")
    with root_engine.begin() as connection:
        connection.execute(text(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}"))
    logger.info(f"Database '{MYSQL_DATABASE}' is ready!")
    
    # Now connect to the specific database
    connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
    engine = create_engine(connection_string, echo=False)
    
    logger.info("\nAttempting to connect to data warehouse database...")
    with engine.connect() as test_connection:
        test_connection.execute(text("SELECT 1"))
    logger.info("Connected to data warehouse database successfully!")
    
except Exception as e:
    logger.error(f"Failed to connect to MySQL")
    logger.error(f"Error: {e}")
    logger.info("\nPlease verify your MySQL connection settings in the .env file:")
    logger.info(f"  - Host: {MYSQL_HOST}:{MYSQL_PORT}")
    logger.info(f"  - User: {MYSQL_USER}")
    logger.info("\nUpdate the MYSQL_USER and MYSQL_PASSWORD in .env if needed.")
    raise

# Load dimensions to database
logger.info("\n" + "-" * 80)
logger.info("Loading dimension tables to database...")
logger.info("-" * 80)

try:
    dim_location.to_sql('dim_location', engine, if_exists='replace', index=False)
    logger.info(f"  dim_location: {len(dim_location):,} rows loaded")

    dim_time.to_sql('dim_time', engine, if_exists='replace', index=False)
    logger.info(f"  dim_time: {len(dim_time):,} rows loaded")

    dim_weather.to_sql('dim_weather', engine, if_exists='replace', index=False)
    logger.info(f"  dim_weather: {len(dim_weather):,} rows loaded")

    dim_substations.to_sql('dim_substations', engine, if_exists='replace', index=False)
    logger.info(f"  dim_substations: {len(dim_substations):,} rows loaded")
    
except Exception as e:
    logger.error(f"Failed to load dimensions: {e}")
    raise

# Load facts to database
logger.info("\n" + "-" * 80)
logger.info("Loading fact tables to database...")
logger.info("-" * 80)

try:
    fact_consumption.to_sql('fact_consumption', engine, if_exists='replace', index=False)
    logger.info(f" fact_consumption: {len(fact_consumption):,} rows loaded")

    fact_charging.to_sql('fact_charging', engine, if_exists='replace', index=False)
    logger.info(f" fact_charging: {len(fact_charging):,} rows loaded")

    fact_population.to_sql('fact_population', engine, if_exists='replace', index=False)
    logger.info(f" fact_population: {len(fact_population):,} rows loaded")
    
except Exception as e:
    logger.error(f"Failed to load facts: {e}")
    raise

# Verify tables were created
logger.info("\n" + "-" * 80)
logger.info("Verifying tables in database...")
logger.info("-" * 80)

try:
    with engine.connect() as connection:
        result = connection.execute(text(f"""
            SELECT TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_SCHEMA = '{MYSQL_DATABASE}'
            ORDER BY TABLE_NAME
        """))
        tables = [row[0] for row in result]
        logger.info(f"\nTables created in {MYSQL_DATABASE} database:")
        for table in tables:
            logger.info(f"  {table}")
        logger.info(f"\nTotal tables: {len(tables)}")
        
except Exception as e:
    logger.error(f"Failed to verify tables: {e}")
    raise

logger.info("\n" + "=" * 80)
logger.info("STEP COMPLETED SUCCESSFULLY!")
logger.info("=" * 80)

2026-04-04 22:25:11,520 - INFO - ================================================================================
2026-04-04 22:25:11,521 - INFO - STEP 1: LOAD TO DATABASE - Connecting to MySQL and loading data
2026-04-04 22:25:11,522 - INFO - ================================================================================
2026-04-04 22:25:11,525 - INFO - 
MySQL Connection Parameters:
2026-04-04 22:25:11,527 - INFO -   Host: localhost
2026-04-04 22:25:11,529 - INFO -   Port: 3306
2026-04-04 22:25:11,531 - INFO -   Database: data_warehouse
2026-04-04 22:25:11,532 - INFO -   User: root
2026-04-04 22:25:11,533 - INFO - 
Attempting to connect to MySQL server...
2026-04-04 22:25:11,624 - INFO - Connected to MySQL server successfully!
2026-04-04 22:25:11,625 - INFO - 
Checking if database 'data_warehouse' exists...
2026-04-04 22:25:11,649 - INFO - Database 'data_warehouse' is ready!
2026-04-04 22:25:11,650 - INFO - 
Attempting to connect to data warehouse database...
2026-04-04 22:25:11,664 

## Summary

### Dimensional Bus Matrix Implementation

| Data Mart | Star | Dimensions Used |
|-----------|------|------------------|
| Energy | Consumption | Location, Time, Weather, Substations |
| Mobility | Charging | Location, Time |
| Demographics | Population | Location, Time |

### Output Files

**Dimensions:**
- `dimensions/dim_location.csv` - Geographic hierarchy
- `dimensions/dim_time.csv` - Multi-level temporal dimension
- `dimensions/dim_weather.csv` - Daily weather conditions
- `dimensions/dim_substations.csv` - Electrical infrastructure

**Facts:**
- `facts/fact_consumption.csv` - Hourly energy consumption
- `facts/fact_charging.csv` - Monthly EV charging activity
- `facts/fact_population.csv` - Yearly demographic indicators


### All output files are stored to MYSQL database

In [92]:
# Add Indexes - Create indexes for query optimization

logger.info("\n" + "=" * 80)
logger.info("STEP 2: ADD INDEXES - Creating indexes for query optimization")
logger.info("=" * 80)

with engine.begin() as connection:
    # Create indexes on dimension tables (Primary Keys)
    logger.info("\nCreating indexes on Dimension Tables...")
    
    index_commands = [
        # Dimension indexes (Primary Keys)
        "CREATE INDEX idx_dim_location_pk ON DIM_LOCATION(location_id)",
        "CREATE INDEX idx_dim_time_pk ON DIM_TIME(time_id)",
        "CREATE INDEX idx_dim_weather_pk ON DIM_WEATHER(weather_id)",
        "CREATE INDEX idx_dim_substations_pk ON DIM_SUBSTATIONS(substation_id)",
        
        # Fact table indexes (Foreign Keys)
        "CREATE INDEX idx_fact_consumption_location ON FACT_CONSUMPTION(location_id)",
        "CREATE INDEX idx_fact_consumption_hour ON FACT_CONSUMPTION(hour_id)",
        "CREATE INDEX idx_fact_consumption_weather ON FACT_CONSUMPTION(weather_id)",
        "CREATE INDEX idx_fact_consumption_substation ON FACT_CONSUMPTION(substation_id)",
        
        "CREATE INDEX idx_fact_charging_location ON FACT_CHARGING(location_id)",
        "CREATE INDEX idx_fact_charging_month ON FACT_CHARGING(month_id)",
        
        "CREATE INDEX idx_fact_population_location ON FACT_POPULATION(location_id)",
        "CREATE INDEX idx_fact_population_year ON FACT_POPULATION(time_id)",
        
        # Commonly filtered columns in dimensions
        # Note: VARCHAR/TEXT columns require max length specification
        "CREATE INDEX idx_dim_location_municipality ON DIM_LOCATION(municipality_name)",
        "CREATE INDEX idx_dim_location_district ON DIM_LOCATION(district_name)",
        
        "CREATE INDEX idx_dim_time_date ON DIM_TIME(date)",
        "CREATE INDEX idx_dim_time_year ON DIM_TIME(year)",
        "CREATE INDEX idx_dim_time_month ON DIM_TIME(month)",
        
        "CREATE INDEX idx_dim_substations_municipality ON DIM_SUBSTATIONS(municipality_name)",
    ]
    
    for idx_cmd in index_commands:
        try:
            connection.execute(text(idx_cmd))
            logger.info(f"  ✓ {idx_cmd.split('ON')[1].strip()}")
        except Exception as e:
            # MySQL throws error if index already exists without IF NOT EXISTS
            # This is expected behavior on re-runs
            if "Duplicate key name" in str(e) or "already exists" in str(e):
                logger.info(f"  ℹ Index already exists: {idx_cmd.split('ON')[1].strip()}")
            else:
                logger.warning(f"  ✗ {str(e)}")

2026-04-04 22:51:48,091 - INFO - 
2026-04-04 22:51:48,094 - INFO - STEP 2: ADD INDEXES - Creating indexes for query optimization
2026-04-04 22:51:48,097 - INFO - ================================================================================
2026-04-04 22:51:48,100 - INFO - 
Creating indexes on Dimension Tables...
2026-04-04 22:51:48,107 - INFO -   ℹ Index already exists: DIM_LOCATI
2026-04-04 22:51:48,113 - INFO -   ℹ Index already exists: DIM_TIME(time_id)
2026-04-04 22:51:48,118 - INFO -   ℹ Index already exists: DIM_WEATHER(weather_id)
2026-04-04 22:51:48,122 - INFO -   ℹ Index already exists: DIM_SUBSTATI
2026-04-04 22:51:48,127 - INFO -   ℹ Index already exists: FACT_C
2026-04-04 22:51:48,132 - INFO -   ℹ Index already exists: FACT_C
2026-04-04 22:51:48,136 - INFO -   ℹ Index already exists: FACT_C
2026-04-04 22:51:48,143 - INFO -   ℹ Index already exists: FACT_C
2026-04-04 22:51:48,146 - INFO -   ℹ Index already exists: FACT_CHARGING(location_id)
2026-04-04 22:51:48,150 - INFO 

In [ ]:
# OLAP Cubes - Build aggregated views for reporting

logger.info("\n" + "=" * 80)
logger.info("OLAP CUBES - Creating aggregated views for reporting")
logger.info("=" * 80)

# Create aggregated views for OLAP analysis
aggregated_views = []

logger.info("\nCreating aggregated views...")

# View 1: Hourly Energy Consumption by Location (FACT is HOURLY, so hours view is detailed)
# Merge with dim_time to get day_id for daily aggregation
hourly_consumption = fact_consumption.merge(
    dim_time[['hour_id', 'day_id']], on='hour_id', how='left'
).groupby(['day_id', 'location_id']).agg({
    'energy_consumed': 'sum',
    'hour_id': 'count'
}).rename(columns={'hour_id': 'hours_recorded'}).reset_index()
hourly_consumption.to_sql('view_hourly_consumption_by_location', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_hourly_consumption_by_location: {len(hourly_consumption)} rows")
aggregated_views.append(('view_hourly_consumption_by_location', len(hourly_consumption)))

# View 2: Monthly EV Charging by District
monthly_charging_district = fact_charging.merge(
    dim_location[['location_id', 'district_name']], on='location_id'
).groupby(['month_id', 'district_name']).agg({
    'energy_consumed': 'mean',
    'location_id': 'count'
}).rename(columns={'location_id': 'locations_count'}).reset_index()
monthly_charging_district.to_sql('view_monthly_charging_by_district', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_monthly_charging_by_district: {len(monthly_charging_district)} rows")
aggregated_views.append(('view_monthly_charging_by_district', len(monthly_charging_district)))

# View 3: Weather Impact on Energy Consumption
weather_consumption = fact_consumption.merge(
    dim_weather[['weather_id', 'avg_temperature', 'precipitation']], on='weather_id'
).groupby(['avg_temperature', 'precipitation']).agg({
    'energy_consumed': ['sum', 'mean', 'count']
}).reset_index()
weather_consumption.columns = ['avg_temperature', 'precipitation', 'total_consumption', 'avg_consumption', 'records']
weather_consumption.to_sql('view_weather_consumption_impact', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_weather_consumption_impact: {len(weather_consumption)} rows")
aggregated_views.append(('view_weather_consumption_impact', len(weather_consumption)))

# View 4: Yearly Demographic Trends
yearly_demographics = fact_population.merge(
    dim_location[['location_id', 'municipality_name']], on='location_id'
).merge(
    dim_time[['time_id', 'year']], on='time_id'
).groupby(['year', 'municipality_name']).agg({
    'density': 'mean',
    'growth_rate': 'mean'
}).reset_index()
yearly_demographics.to_sql('view_yearly_demographic_trends', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_yearly_demographic_trends: {len(yearly_demographics)} rows")
aggregated_views.append(('view_yearly_demographic_trends', len(yearly_demographics)))

logger.info(f"\nCreated {len(aggregated_views)} aggregated views")
logger.info("✓ Step completed successfully!")

2026-04-04 22:31:01,630 - INFO - 
2026-04-04 22:31:01,631 - INFO - STEP 3: OLAP CUBES - Creating aggregated views for reporting
2026-04-04 22:31:01,633 - INFO - ================================================================================
2026-04-04 22:31:01,634 - INFO - 
Creating aggregated views...
2026-04-04 22:31:04,541 - INFO -   ✓ view_hourly_consumption_by_location: 47450 rows
2026-04-04 22:31:04,677 - INFO -   ✓ view_monthly_charging_by_district: 96 rows
2026-04-04 22:31:06,223 - INFO -   ✓ view_weather_consumption_impact: 330 rows
2026-04-04 22:31:06,934 - INFO -   ✓ view_yearly_demographic_trends: 68 rows
2026-04-04 22:31:06,935 - INFO - 
Created 4 aggregated views
2026-04-04 22:31:06,936 - INFO - ✓ Step completed successfully!


In [87]:
# Test Queries - Run dimensional analysis queries

logger.info("\n" + "=" * 80)
logger.info("TEST QUERIES - Running dimensional analysis queries")
logger.info("=" * 80)

query_results = []

# Query 1: Total consumption by district
query1 = """
SELECT 
    dl.district_name,
    SUM(fc.energy_consumed) as total_consumption,
    AVG(fc.energy_consumed) as avg_consumption,
    COUNT(*) as records
FROM FACT_CONSUMPTION fc
JOIN DIM_LOCATION dl ON fc.location_id = dl.location_id
GROUP BY dl.district_name
ORDER BY total_consumption DESC;
"""
logger.info("\nQuery 1: Total consumption by district")
result1 = pd.read_sql_query(query1, engine)
logger.info(f"\nResults ({len(result1)} districts):")
print(result1.to_string())
query_results.append(('Q1_Consumption_by_District', result1))

# Query 2: Peak consumption hours
query2 = """
SELECT 
    dt.hour,
    AVG(fc.energy_consumed) as avg_consumption,
    MAX(fc.energy_consumed) as max_consumption,
    COUNT(*) as records
FROM FACT_CONSUMPTION fc
JOIN DIM_TIME dt ON fc.hour_id = dt.hour_id
GROUP BY dt.hour
ORDER BY avg_consumption DESC;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 2: Peak consumption hours")
result2 = pd.read_sql_query(query2, engine)
logger.info(f"\nTop 5 peak hours:")
print(result2.head(5).to_string())
query_results.append(('Q2_Peak_Hours', result2))

# Query 3: Weather correlation with consumption
query3 = """
SELECT 
    dw.avg_temperature as temperature,
    ROUND(dw.precipitation, 2) as precipitation,
    AVG(fc.energy_consumed) as avg_consumption,
    COUNT(*) as sample_size
FROM FACT_CONSUMPTION fc
JOIN DIM_WEATHER dw ON fc.weather_id = dw.weather_id
GROUP BY dw.avg_temperature, dw.precipitation
ORDER BY avg_consumption DESC
LIMIT 10;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 3: Weather correlation with consumption")
result3 = pd.read_sql_query(query3, engine)
logger.info(f"\nTop conditions:")
print(result3.to_string())
query_results.append(('Q3_Weather_Correlation', result3))

# Query 4: Charging activity by municipality and outlet type
query4 = """
SELECT 
    dl.municipality_name,
    fch.outlet_type,
    SUM(fch.energy_consumed) as total_energy,
    AVG(fch.energy_consumed) as avg_energy,
    COUNT(*) as records
FROM FACT_CHARGING fch
JOIN DIM_LOCATION dl ON fch.location_id = dl.location_id
GROUP BY dl.municipality_name, fch.outlet_type
ORDER BY total_energy DESC
LIMIT 10;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 4: Top charging municipalities by outlet type")
result4 = pd.read_sql_query(query4, engine)
logger.info(f"\nTop 10 results:")
print(result4.to_string())
query_results.append(('Q4_Top_Charging_Activity', result4))

# Query 5: Demographics statistics by district
query5 = """
SELECT 
    dl.district_name,
    COUNT(*) as location_count,
    AVG(fp.density) as avg_density,
    AVG(fp.growth_rate) as avg_growth_rate
FROM FACT_POPULATION fp
JOIN DIM_LOCATION dl ON fp.location_id = dl.location_id
GROUP BY dl.district_name
ORDER BY avg_density DESC;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 5: Demographics statistics by district")
result5 = pd.read_sql_query(query5, engine)
logger.info(f"\nResults ({len(result5)} districts):")
print(result5.to_string())
query_results.append(('Q5_Demographics_Statistics', result5))

logger.info("\n" + "=" * 80)
logger.info(f"✓ Step completed successfully! ({len(query_results)} queries executed)")

2026-04-04 22:31:10,971 - INFO - 
2026-04-04 22:31:10,972 - INFO - TEST QUERIES - Running dimensional analysis queries
2026-04-04 22:31:10,973 - INFO - ================================================================================
2026-04-04 22:31:10,975 - INFO - 
Query 1: Total consumption by district
2026-04-04 22:38:06,726 - INFO - 
Results (2 districts):
2026-04-04 22:38:06,732 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 22:38:06,734 - INFO - Query 2: Peak consumption hours


  district_name  total_consumption  avg_consumption  records
0         Porto       2.336347e+08        35.888406  6510033
1        Aveiro       3.065371e+07         6.593271  4649242


2026-04-04 22:44:01,316 - INFO - 
Top 5 peak hours:
2026-04-04 22:44:01,321 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 22:44:01,322 - INFO - Query 3: Weather correlation with consumption


   hour  avg_consumption  max_consumption  records
0    13        36.339060           976.23   464592
1    12        36.283175           917.37   465671
2    16        36.129434           866.57   465738
3    18        36.012864          1053.98   465069
4    19        35.699365           809.70   465308


2026-04-04 22:49:54,583 - INFO - 
Top conditions:
2026-04-04 22:49:54,588 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 22:49:54,589 - INFO - Query 4: Top charging municipalities by outlet type


   temperature  precipitation  avg_consumption  sample_size
0        10.10           17.2        34.190897        31062
1        10.35           20.7        32.654402        30977
2         8.65           40.3        32.264504        30676
3         6.55           24.3        32.206787        31148
4         9.65            0.0        31.537173        31993
5         7.95           19.7        31.515620        30699
6        11.15           62.2        31.367211        30574
7         7.90            0.0        31.077816        32036
8         8.95           50.8        30.149855        29386
9         5.90            0.0        29.863469        30915


2026-04-04 22:49:54,947 - INFO - 
Top 10 results:
2026-04-04 22:49:54,950 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 22:49:54,950 - INFO - Query 5: Demographics statistics by district
2026-04-04 22:49:54,962 - INFO - 
Results (2 districts):
2026-04-04 22:49:54,964 - INFO - 
2026-04-04 22:49:54,965 - INFO - ✓ Step completed successfully! (5 queries executed)


      municipality_name outlet_type  total_energy  avg_energy  records
0     Vila Nova de Gaia       Total      149040.0  207.000000      720
1                 Porto       Total      110348.0  328.416667      336
2     Vila Nova de Gaia    Mennekes       99330.0  137.958333      720
3  Santa Maria da Feira       Total       77910.0   77.291667     1008
4                 Porto    Mennekes       77147.0  229.604167      336
5  Santa Maria da Feira    Mennekes       51387.0   50.979167     1008
6            Matosinhos       Total       36192.0  188.500000      192
7                  Maia       Total       35856.0   83.000000      432
8     Vila Nova de Gaia         CCS       26880.0   37.333333      720
9            Matosinhos    Mennekes       24640.0  128.333333      192
  district_name  location_count  avg_density  avg_growth_rate
0         Porto             356  1506.384551         1.024045
1        Aveiro             164   606.832927         0.441585


In [88]:
# Documentation - Create data dictionary and business glossary

logger.info("\n" + "=" * 80)
logger.info(" DOCUMENTATION - Creating data dictionary and business glossary")
logger.info("=" * 80)

# Initialize documentation
data_dictionary = []
business_glossary = []

# Data Dictionary - Dimensions
logger.info("\nGenerating data dictionary for dimensions...")

dimensions_doc = {
    'DIM_LOCATION': {
        'description': 'Geographic hierarchy of Porto Metropolitan Area',
        'columns': dim_location.columns.tolist(),
        'row_count': len(dim_location),
        'grain': 'One row per municipality with geographic hierarchies'
    },
    'DIM_TIME': {
        'description': 'Multi-level temporal dimension with hourly granularity',
        'columns': dim_time.columns.tolist(),
        'row_count': len(dim_time),
        'grain': 'One row per hour from min_date to max_date'
    },
    'DIM_WEATHER': {
        'description': 'Daily weather conditions from two monitoring stations',
        'columns': dim_weather.columns.tolist(),
        'row_count': len(dim_weather),
        'grain': 'One row per day with aggregated weather metrics'
    },
    'DIM_SUBSTATIONS': {
        'description': 'Electrical infrastructure substation details',
        'columns': dim_substations.columns.tolist(),
        'row_count': len(dim_substations),
        'grain': 'One row per unique substation'
    }
}

# Data Dictionary - Facts
facts_doc = {
    'FACT_CONSUMPTION': {
        'description': 'Hourly energy consumption by municipality',
        'columns': fact_consumption.columns.tolist(),
        'row_count': len(fact_consumption),
        'grain': 'One row per municipality per hour',
        'measures': ['energy_consumed']
    },
    'FACT_CHARGING': {
        'description': 'Monthly EV charging activity and outlet types',
        'columns': fact_charging.columns.tolist(),
        'row_count': len(fact_charging),
        'grain': 'One row per municipality per month per outlet type',
        'measures': ['energy_consumed']
    },
    'FACT_POPULATION': {
        'description': 'Yearly demographic indicators by municipality',
        'columns': fact_population.columns.tolist(),
        'row_count': len(fact_population),
        'grain': 'One row per municipality per year',
        'measures': ['density', 'growth_rate']
    }
}

# Create comprehensive documentation
documentation_content = f"""
# DATA WAREHOUSE DOCUMENTATION
## Porto Metropolitan Area - Energy, Mobility & Demographics Data Warehouse

Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## 1. EXECUTIVE SUMMARY

This data warehouse integrates energy consumption, electric vehicle charging infrastructure, and demographic data for the Porto Metropolitan Area. The schema follows a star topology with conformed dimensions enabling analysis across multiple data marts.

### Data Warehouse Dimensions
- **Geographic**: Municipality-level analysis with regional/district hierarchies
- **Temporal**: Hourly granularity with multi-level hierarchies (hour, day, month, year)
- **Weather**: Daily weather conditions from two monitoring stations
- **Infrastructure**: Electrical distribution substation details

### Data Warehouse Facts
- **FACT_CONSUMPTION**: {len(fact_consumption):,} hourly energy consumption records
- **FACT_CHARGING**: {len(fact_charging):,} EV charging activity records
- **FACT_POPULATION**: {len(fact_population):,} yearly demographic records

---

## 2. DIMENSIONAL BUS MATRIX

| Data Mart | Fact Table | Dimensions |
|-----------|-----------|-----------|
| Energy | FACT_CONSUMPTION | Location, Time, Weather, Substations |
| Mobility | FACT_CHARGING | Location, Time |
| Demographics | FACT_POPULATION | Location, Time |

---

## 3. DIMENSION DOCUMENTATION

### DIM_LOCATION
**Purpose**: {dimensions_doc['DIM_LOCATION']['description']}
**Grain**: {dimensions_doc['DIM_LOCATION']['grain']}
**Row Count**: {dimensions_doc['DIM_LOCATION']['row_count']:,}

**Columns**:
"""

for col in dimensions_doc['DIM_LOCATION']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### DIM_TIME
**Purpose**: {dimensions_doc['DIM_TIME']['description']}
**Grain**: {dimensions_doc['DIM_TIME']['grain']}
**Row Count**: {dimensions_doc['DIM_TIME']['row_count']:,}
**Date Range**: {dim_time['date'].min()} to {dim_time['date'].max()}

**Columns**:
"""

for col in dimensions_doc['DIM_TIME']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### DIM_WEATHER
**Purpose**: {dimensions_doc['DIM_WEATHER']['description']}
**Grain**: {dimensions_doc['DIM_WEATHER']['grain']}
**Row Count**: {dimensions_doc['DIM_WEATHER']['row_count']:,}

**Columns**:
"""

for col in dimensions_doc['DIM_WEATHER']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### DIM_SUBSTATIONS
**Purpose**: {dimensions_doc['DIM_SUBSTATIONS']['description']}
**Grain**: {dimensions_doc['DIM_SUBSTATIONS']['grain']}
**Row Count**: {dimensions_doc['DIM_SUBSTATIONS']['row_count']:,}

**Columns**:
"""

for col in dimensions_doc['DIM_SUBSTATIONS']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

## 4. FACT TABLE DOCUMENTATION

### FACT_CONSUMPTION
**Purpose**: {facts_doc['FACT_CONSUMPTION']['description']}
**Grain**: {facts_doc['FACT_CONSUMPTION']['grain']}
**Row Count**: {facts_doc['FACT_CONSUMPTION']['row_count']:,}
**Measures**: {', '.join(facts_doc['FACT_CONSUMPTION']['measures'])}

**Columns**:
"""

for col in facts_doc['FACT_CONSUMPTION']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### FACT_CHARGING
**Purpose**: {facts_doc['FACT_CHARGING']['description']}
**Grain**: {facts_doc['FACT_CHARGING']['grain']}
**Row Count**: {facts_doc['FACT_CHARGING']['row_count']:,}
**Measures**: {', '.join(facts_doc['FACT_CHARGING']['measures'])}

**Columns**:
"""

for col in facts_doc['FACT_CHARGING']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### FACT_POPULATION
**Purpose**: {facts_doc['FACT_POPULATION']['description']}
**Grain**: {facts_doc['FACT_POPULATION']['grain']}
**Row Count**: {facts_doc['FACT_POPULATION']['row_count']:,}
**Measures**: {', '.join(facts_doc['FACT_POPULATION']['measures'])}

**Columns**:
"""

for col in facts_doc['FACT_POPULATION']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

## 5. BUSINESS GLOSSARY

### Energy & Infrastructure Terms
- **Energy Consumed**: Electrical power consumption measured in kilowatt-hours (kWh)
- **Substation**: Electrical distribution point for power transmission
- **Installation Code**: Unique identifier for electrical infrastructure
- **District**: Administrative district in Porto Metropolitan Area

### Mobility Terms
- **Outlet Type**: Classification of EV charging outlets (e.g., Total, DC, AC)
- **Charging Energy**: Energy provided by charging infrastructure

### Geographic Terms
- **Municipality**: Administrative subdivision of Porto Metropolitan Area
- **District**: Grouping of municipalities for analysis
- **Parish**: Sub-division of municipality for geographic analysis

### Demographic Terms
- **Density**: Population density (people per square km)
- **Growth Rate**: Year-over-year population growth percentage

### Temporal Terms
- **Hour ID**: Unique identifier for each hour in the temporal dimension
- **Day ID**: Unique identifier for each day in the temporal dimension
- **Month ID**: Unique identifier for each month in the temporal dimension
- **Time ID**: Surrogate key for time dimension records

### Weather Terms
- **Average Temperature**: Mean temperature for the day (°C)
- **Minimum Temperature**: Lowest temperature recorded for the day (°C)
- **Maximum Temperature**: Highest temperature recorded for the day (°C)
- **Precipitation**: Total rainfall for the day (mm)
- **Wind Speed**: Average wind speed for the day (km/h)
- **Pressure**: Atmospheric pressure (hPa)

---

## 6. AGGREGATED VIEWS

The following views have been created for reporting:
- `VIEW_DAILY_CONSUMPTION_BY_LOCATION`: Daily energy totals by municipality
- `VIEW_MONTHLY_CHARGING_BY_DISTRICT`: Monthly charging activity by district
- `VIEW_WEATHER_CONSUMPTION_IMPACT`: Consumption patterns by weather conditions
- `VIEW_YEARLY_DEMOGRAPHIC_TRENDS`: Demographic trends by municipality

---

## 7. DATA QUALITY METRICS

**Completeness**:
- DIM_LOCATION: 100% ({len(dim_location)} municipalities)
- DIM_TIME: {(~dim_time['datetime'].isna()).sum() / len(dim_time) * 100:.1f}% ({(~dim_time['datetime'].isna()).sum():,} records)
- DIM_WEATHER: {(~dim_weather['weather_id'].isna()).sum() / len(dim_weather) * 100:.1f}% ({(~dim_weather['weather_id'].isna()).sum():,} records)
- DIM_SUBSTATIONS: {(~dim_substations['substation_id'].isna()).sum() / len(dim_substations) * 100:.1f}% ({(~dim_substations['substation_id'].isna()).sum():,} records)

**Grain Validation**:
- FACT_CONSUMPTION: {len(fact_consumption):,} hourly records (expected granularity maintained)
- FACT_CHARGING: {len(fact_charging):,} records (expected granularity maintained)
- FACT_POPULATION: {len(fact_population):,} yearly records (expected granularity maintained)

---

## 8. DATABASE ARTIFACTS

**Database Location**: MySQL - {MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}
**Database Type**: MySQL 8.0+
**Database User**: {MYSQL_USER}

**Tables Created**: 
- 7 (4 dimensions + 3 facts)

**Indexes Created**: 17
- Primary key indexes for all dimensions
- Foreign key indexes for all facts
- Commonly queried column indexes

---

## 9. ETL PROCESS SUMMARY

**Extract Phase**: 4 source datasets processed
**Transform Phase**: Data cleaning, aggregation, and standardization
**Load Phase**: Data loaded to MySQL database with full integrity
**Validation Phase**: 7 data quality checks performed
**Reporting Phase**: Aggregated views created

---

## 10. CONTACT & SUPPORT

For questions about this data warehouse, please contact the data engineering team.
Documentation Version: 1.0
Last Updated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

# Save documentation to file
doc_path = os.path.join(OUTPUT_DIR, 'DATA_WAREHOUSE_DOCUMENTATION.md')
with open(doc_path, 'w', encoding='utf-8') as f:
    f.write(documentation_content)
logger.info(f"✓ Data dictionary and documentation saved to: {doc_path}")

# Save data dictionary as CSV
data_dict_summary = pd.DataFrame([
    {'Table': k, 'Type': 'Dimension', 'Description': v['description'], 'Row Count': v['row_count']}
    for k, v in dimensions_doc.items()
] + [
    {'Table': k, 'Type': 'Fact', 'Description': v['description'], 'Row Count': v['row_count']}
    for k, v in facts_doc.items()
])

dict_path = os.path.join(OUTPUT_DIR, 'data_dictionary_summary.csv')
data_dict_summary.to_csv(dict_path, index=False)
logger.info(f"✓ Data dictionary summary saved to: {dict_path}")

logger.info("\n" + "=" * 80)
logger.info("✓ ALL STEPS COMPLETED SUCCESSFULLY!")
logger.info("=" * 80)
logger.info("\nData Warehouse is now ready for analysis and reporting!")

2026-04-04 22:50:12,189 - INFO - 
2026-04-04 22:50:12,191 - INFO -  DOCUMENTATION - Creating data dictionary and business glossary
2026-04-04 22:50:12,193 - INFO - ================================================================================
2026-04-04 22:50:12,195 - INFO - 
Generating data dictionary for dimensions...
2026-04-04 22:50:12,212 - INFO - ✓ Data dictionary and documentation saved to: ./dw_output\DATA_WAREHOUSE_DOCUMENTATION.md
2026-04-04 22:50:12,224 - INFO - ✓ Data dictionary summary saved to: ./dw_output\data_dictionary_summary.csv
2026-04-04 22:50:12,226 - INFO - 
2026-04-04 22:50:12,228 - INFO - ✓ ALL STEPS COMPLETED SUCCESSFULLY!
2026-04-04 22:50:12,229 - INFO - ================================================================================
2026-04-04 22:50:12,231 - INFO - 
Data Warehouse is now ready for analysis and reporting!


In [89]:
# DEBUG: Check actual column names
print("=" * 80)
print("DEBUG: Checking DataFrame columns")
print("=" * 80)
print("\nfact_consumption columns:")
print(fact_consumption.columns.tolist())
print(f"Shape: {fact_consumption.shape}")
print("\nfact_consumption head:")
print(fact_consumption.head())

print("\n\ndim_location columns:")
print(dim_location.columns.tolist())
print(f"Shape: {dim_location.shape}")
print("\ndim_location head:")
print(dim_location.head())

print("\n\nfact_charging columns:")
print(fact_charging.columns.tolist())
print(f"Shape: {fact_charging.shape}")
print("\nfact_charging head:")
print(fact_charging.head())


DEBUG: Checking DataFrame columns

fact_consumption columns:
['location_id', 'hour_id', 'weather_id', 'substation_id', 'energy_consumed']
Shape: (11159275, 5)

fact_consumption head:
   location_id  hour_id  weather_id  substation_id  energy_consumed
0            1    41620        1370             25              0.0
1            2    41620        1370             25              0.0
2            4    41620        1370             25              0.0
3            7    41620        1370             25              0.0
4            8    41620        1370             25              0.0


dim_location columns:
['location_id', 'municipality_code', 'municipality_name', 'parish_code', 'parish_name', 'district_code', 'district_name']
Shape: (130, 7)

dim_location head:
   location_id municipality_code     municipality_name parish_code  \
0            1           11A0109  Santa Maria da Feira      010914   
1            2           11A0109  Santa Maria da Feira      010904   
2            3   